# Phase 2 — Train Transformer dự đoán Cluster-SID

Notebook này train `EncoderDecoderRetrievalModel` trong `train_decoder.py`. Tên file source là decoder, nhưng mô hình được train ở đây chính là T5 Encoder–Decoder Transformer cho sequential recommendation.

Luồng dữ liệu:

```text
prev_items → product_index → cluster SID sequence → Transformer → next cluster SID
```

Notebook dùng trực tiếp `semantic_ids.parquet` từ notebook 03; không re-encode embedding và không cần load checkpoint RQ-VAE. Loss gồm ba cross-entropy tương ứng codebook `[128, 64, 32]`. Metric ở bước này là SID/cluster-level Hit và NDCG, chưa phải item-level ranking metric.

## Trước khi chạy

1. Bật GPU trong Kaggle Notebook Settings.
2. Add Input là output `preprocessed` của notebook 01.
3. Add Input là output `vmarket_rqvae` của notebook 03.
4. Tạo Kaggle Secret `GITHUB_TOKEN` có quyền đọc repository.
5. Tạo Kaggle Secret `WANDB_API_KEY`.
6. Các đường dẫn input và output lấy từ `configs/transformer_vmarket.gin`.

## 0. Cấu hình

In [ ]:
from pathlib import Path

GITHUB_REPOSITORY_URL = "https://github.com/nam-htran/VSF-MiniApp-Ecommerce.git"
GITHUB_BRANCH = "main"
REPOSITORY_ROOT = Path("/kaggle/working/vsf-miniapp-ecommerce-source")
AUTO_INSTALL_DEPENDENCIES = True

print("Configuration loaded.")

## 1. Cài dependency và kiểm tra GPU

In [ ]:
import importlib.metadata as metadata
import subprocess
import sys

from packaging.version import Version


requirements = {
    "gin-config": "0.5.0",
    "accelerate": "1.0.0",
    "einops": "0.8.0",
    "transformers": "4.46.0",
    "wandb": "0.19.0",
    "pyarrow": "16.0.0",
}
packages_to_install = []
for distribution, minimum_version in requirements.items():
    try:
        installed_version = metadata.version(distribution)
    except metadata.PackageNotFoundError:
        installed_version = None
    if installed_version is None or Version(installed_version) < Version(minimum_version):
        packages_to_install.append(f"{distribution}>={minimum_version}")

if AUTO_INSTALL_DEPENDENCIES and packages_to_install:
    print("Installing:", packages_to_install)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *packages_to_install])

import pandas as pd
import torch

if Version(torch.__version__.split("+")[0]) < Version("2.5.0"):
    raise RuntimeError(f"PyTorch >= 2.5.0 is required, found {torch.__version__}")

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Transformers:", metadata.version("transformers"))
print("CUDA available:", torch.cuda.is_available())
for index in range(torch.cuda.device_count()):
    properties = torch.cuda.get_device_properties(index)
    print(f"cuda:{index}: {properties.name}, {properties.total_memory / 2**30:.1f} GiB")
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator before training the Transformer.")

## 2. Kết nối Weights & Biases

Đọc `WANDB_API_KEY` từ Kaggle Secret và đăng nhập W&B.

In [ ]:
import os


from kaggle_secrets import UserSecretsClient

os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")

## 3. Clone source từ GitHub

Đọc `GITHUB_TOKEN` từ Kaggle Secret và clone nhánh `main`.

In [ ]:
from kaggle_secrets import UserSecretsClient


github_token = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPOSITORY_ROOT = Path(REPOSITORY_ROOT).expanduser().resolve()

git_environment = {
    **os.environ,
    "GITHUB_TOKEN": github_token,
    "GIT_TERMINAL_PROMPT": "0",
}
credential_helper = "!f() { echo username=x-access-token; echo password=$GITHUB_TOKEN; }; f"
git = ["git", "-c", f"credential.helper={credential_helper}"]

if (REPOSITORY_ROOT / ".git").is_dir():
    subprocess.run(
        [*git, "-C", str(REPOSITORY_ROOT), "pull", "--ff-only", "origin", GITHUB_BRANCH],
        check=True,
        env=git_environment,
    )
elif REPOSITORY_ROOT.exists():
    raise FileExistsError(f"Clone target is not a Git repository: {REPOSITORY_ROOT}")
else:
    subprocess.run(
        [*git, "clone", "--depth", "1", "--branch", GITHUB_BRANCH, GITHUB_REPOSITORY_URL, str(REPOSITORY_ROOT)],
        check=True,
        env=git_environment,
    )
del github_token, git_environment

SOURCE_ROOT = REPOSITORY_ROOT / "ai-recommendation/src"
if not (SOURCE_ROOT / "train_decoder.py").is_file():
    raise FileNotFoundError(f"Transformer source not found: {SOURCE_ROOT}")
print("SOURCE_ROOT:", SOURCE_ROOT)

## 4. Chọn Gin config

In [ ]:
CONFIG_PATH = SOURCE_ROOT / "configs/transformer_vmarket.gin"
if not CONFIG_PATH.is_file():
    raise FileNotFoundError(CONFIG_PATH)

print("Gin config:", CONFIG_PATH)

## 5. Train Transformer

In [ ]:
command = [sys.executable, "train_decoder.py", str(CONFIG_PATH)]
print("Running:", " ".join(command))
subprocess.run(command, cwd=SOURCE_ROOT, check=True)

## 6. Kiểm tra output

In [ ]:
import json

OUTPUT_ROOT = Path("/kaggle/working/vmarket_transformer")
SESSION_CACHE_ROOT = Path("/kaggle/working/vmarket_transformer_session_cache")
metrics_path = OUTPUT_ROOT / "transformer_metrics.json"

In [ ]:
if not metrics_path.is_file():
    raise FileNotFoundError(f"Transformer metrics not found: {metrics_path}")
checkpoints = sorted(OUTPUT_ROOT.glob("checkpoint_*.pt"))
if not checkpoints:
    raise FileNotFoundError("No Transformer checkpoint was written.")
metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
cache_manifest_path = SESSION_CACHE_ROOT / "session_cache_manifest.json"
if not cache_manifest_path.is_file():
    raise FileNotFoundError("Session cache manifest was not written.")
cache_manifest = json.loads(cache_manifest_path.read_text(encoding="utf-8"))
output_size = sum(path.stat().st_size for path in OUTPUT_ROOT.rglob("*") if path.is_file())

print("Final Transformer validation: PASSED")
print("Latest checkpoint:", checkpoints[-1])
print("Output size:", f"{output_size / 2**30:.2f} GiB")
print("Truncated sessions:", cache_manifest["truncated_sessions"])
display(pd.DataFrame([metrics]).T.rename(columns={0: "value"}))
print("OUTPUT_ROOT:", OUTPUT_ROOT)

## Kết quả của bước này

Notebook hoàn thành khi cell cuối báo `Final Transformer validation: PASSED`. Output là Transformer có thể sinh Top-K cluster SID. Bước tiếp theo mới mở rộng cluster thành candidate item và đánh giá/ranking ở item level.